In [1]:
import pandas as pd
import arcpy
import requests

## Bring in scenic data from 

In [2]:
scenic_points = r'F:\GIS\DB_CONNECT\Collection.sde\SDE.Scenic\SDE.Scenic_Photo_Point'
scenic_photos = r'F:\GIS\DB_CONNECT\Collection.sde\SDE.Scenic_Photo_Point__ATTACH'

# output location for the joined table
gdb_path = r"C:\GIS\Scratch.gdb"
output_table = "Scenic_Photo_Attachments_Joined"
output_path = f"{gdb_path}\\{output_table}"

## Parent data class is scenic points populate scenic photos based on REL_GLOBALID TO GLOBABLID

In [9]:
# Build a GlobalID -> (ID, UNIT, CATEGORY) lookup from the scenic points (parent) layer
points_lookup = {}
with arcpy.da.SearchCursor(scenic_points, ["GlobalID", "ID", "UNIT", "CATEGORY"]) as cursor:
    for global_id, id_val, unit, category in cursor:
        points_lookup[global_id] = (id_val, unit, category)

print(f"Loaded {len(points_lookup)} scenic points")

Loaded 392 scenic points


In [10]:
# Create the output table that will hold scenic photo attachments joined to their parent point's ID/UNIT/CATEGORY
arcpy.env.overwriteOutput = True

if arcpy.Exists(output_path):
    arcpy.management.Delete(output_path)
    print(f"Deleted existing table: {output_table}")

arcpy.management.CreateTable(gdb_path, output_table)

arcpy.management.AddField(output_path, "SOURCE_ATTACHMENTID", "LONG")
arcpy.management.AddField(output_path, "REL_GLOBALID", "GUID")
arcpy.management.AddField(output_path, "CONTENT_TYPE", "TEXT", field_length=150)
arcpy.management.AddField(output_path, "ATT_NAME", "TEXT", field_length=250)
arcpy.management.AddField(output_path, "DATA_SIZE", "LONG")
arcpy.management.AddField(output_path, "DATA", "BLOB")
arcpy.management.AddField(output_path, "PhotoID", "TEXT", field_length=255)
arcpy.management.AddField(output_path, "ID", "TEXT", field_length=255)
arcpy.management.AddField(output_path, "UNIT", "TEXT", field_length=255)
arcpy.management.AddField(output_path, "CATEGORY", "TEXT", field_length=255)

print(f"Created table: {output_path}")

Deleted existing table: Scenic_Photo_Attachments_Joined
Created table: C:\GIS\Scratch.gdb\Scenic_Photo_Attachments_Joined


In [11]:
# Read scenic photo attachments (child) and write each row joined to its parent point's ID/UNIT/CATEGORY
attach_fields = ["ATTACHMENTID", "REL_GLOBALID", "CONTENT_TYPE", "ATT_NAME", "DATA_SIZE", "DATA", "PhotoID"]
out_fields = ["SOURCE_ATTACHMENTID", "REL_GLOBALID", "CONTENT_TYPE", "ATT_NAME", "DATA_SIZE", "DATA", "PhotoID", "ID", "UNIT", "CATEGORY"]

unmatched = 0
written = 0

with arcpy.da.SearchCursor(scenic_photos, attach_fields) as search_cursor, \
     arcpy.da.InsertCursor(output_path, out_fields) as insert_cursor:
    for attachment_id, rel_global_id, content_type, att_name, data_size, data, photo_id in search_cursor:
        id_val, unit, category = points_lookup.get(rel_global_id, (None, None, None))
        if id_val is None and unit is None and category is None:
            unmatched += 1
        insert_cursor.insertRow((attachment_id, rel_global_id, content_type, att_name, data_size, data, photo_id, id_val, unit, category))
        written += 1

print(f"Wrote {written} photo attachment records ({unmatched} with no matching scenic point) to {output_path}")

Wrote 443 photo attachment records (0 with no matching scenic point) to C:\GIS\Scratch.gdb\Scenic_Photo_Attachments_Joined


In [12]:
# For Roadway category: append "-roadway" to ID if not already present, and set ATT_NAME to ID + "-2023.jpg"
updated = 0
with arcpy.da.UpdateCursor(output_path, ["CATEGORY", "ID", "ATT_NAME"]) as cursor:
    for category, id_val, att_name in cursor:
        if category == "Roadway" and id_val and "-roadway" not in id_val:
            new_id = f"{id_val}-roadway"
            new_att_name = f"{new_id}-2023.jpg"
            cursor.updateRow([category, new_id, new_att_name])
            updated += 1

print(f"Updated {updated} Roadway records with -roadway suffix and ATT_NAME")

Updated 262 Roadway records with -roadway suffix and ATT_NAME


In [13]:
sde_scenic_points = r'F:\GIS\DB_CONNECT\Vector.sde\SDE.Scenic\SDE.Scenic_Viewpoint'


In [14]:
# Build a lookup from ID to GlobalID from the scenic viewpoint feature class
id_to_scenic_global_id = {}
with arcpy.da.SearchCursor(sde_scenic_points, ["ID", "GlobalID"]) as cursor:
    for id_val, global_id in cursor:
        if id_val:
            id_to_scenic_global_id[id_val] = global_id

print(f"Built lookup with {len(id_to_scenic_global_id)} scenic viewpoint IDs")

# Add Scenic_Viewpoint_GlobalID field if it doesn't exist
fields = [f.name for f in arcpy.ListFields(output_path)]
if "Scenic_Viewpoint_GlobalID" not in fields:
    arcpy.management.AddField(output_path, "Scenic_Viewpoint_GlobalID", "GUID")
    print("Added Scenic_Viewpoint_GlobalID field")

# Join the scenic viewpoint GlobalID to the attachments table based on ID
joined = 0
unmatched_ids = 0
with arcpy.da.UpdateCursor(output_path, ["ID", "Scenic_Viewpoint_GlobalID"]) as cursor:
    for id_val, scenic_global_id in cursor:
        if id_val and id_val in id_to_scenic_global_id:
            new_scenic_global_id = id_to_scenic_global_id[id_val]
            cursor.updateRow([id_val, new_scenic_global_id])
            joined += 1
        elif id_val:
            unmatched_ids += 1

print(f"Populated Scenic_Viewpoint_GlobalID for {joined} records ({unmatched_ids} attachment IDs had no match in scenic viewpoints)")

Built lookup with 396 scenic viewpoint IDs
Added Scenic_Viewpoint_GlobalID field
Populated Scenic_Viewpoint_GlobalID for 382 records (60 attachment IDs had no match in scenic viewpoints)
